In [ ]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/GoogleColab/EnChatbot"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


import os

print(os.path.exists(PROJECT_ROOT))

True


In [ ]:
# downloading necessary libraries

!pip install -r /content/drive/MyDrive/GoogleColab/EnChatbot/requirement
!pip install -q \
transformers==4.56.1 \
datasets \
accelerate \
sentencepiece \
evaluate \
sacrebleu

In [ ]:
# importing libraries

import os
import sys
import pandas as pd
import numpy as np
import tensorflow as tf
import torch
from tensorflow import keras

sys.path.append("/content/drive/MyDrive/GoogleColab/EnChatbot/backend")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# checking datasets availability

from gec_datasets import GECDatasets
gec = GECDatasets(
    base_path="gec_dataset_base",
)

import gec_datasets
print(gec_datasets.available())

['conll13', 'conll14', 'bea19-dev', 'bea19-test', 'wi-locness-train', 'jfleg-dev', 'jfleg-test', 'cweb-g-dev', 'cweb-g-test', 'cweb-s-dev', 'cweb-s-test', 'fce-train', 'fce-dev', 'fce-test', 'nucle-train', 'lang8-train', 'troy-1bw-train', 'troy-1bw-dev', 'troy-blogs-train', 'troy-blogs-dev', 'pie-synthetic-a1', 'pie-synthetic-a2', 'pie-synthetic-a3', 'pie-synthetic-a4', 'pie-synthetic-a5']


In [ ]:
# importing database(s)

import shutil

!mkdir "gec_dataset_base/lang8"
shutil.copy(
    "/content/drive/MyDrive/GoogleColab/EnChatbot/datasets/lang-8/lang8.bea19.tar.gz",
    "gec_dataset_base/lang8/lang8.bea19.tar.gz"
)

mkdir: cannot create directory ‘gec_dataset_base/lang8’: File exists


'gec_dataset_base/lang8/lang8.bea19.tar.gz'

In [ ]:
lang8 = gec.load("lang8-train")
lang8_src = lang8.srcs
lang8_tgt = lang8.refs[0]

In [ ]:
bea = gec.load("wi-locness-train")
bea_src = bea.srcs
bea_tgt = bea.refs[0]

In [ ]:
fce = gec.load("fce-train")
fce_src = fce.srcs
fce_tgt = fce.refs[0]

In [ ]:
# Cleaning function

def clean(src, tgt):

    clean_src = []
    clean_tgt = []

    seen = set()

    removed_duplicates = 0
    removed_empty = 0
    removed_short = 0

    for s, t in zip(src, tgt):

        if not s or not t:
            removed_empty += 1
            continue

        s = s.strip()
        t = t.strip()

        if len(s) < 3 or len(t) < 3:
            removed_short += 1
            continue

        pair = (s, t)

        if pair in seen:
            removed_duplicates += 1
            continue

        seen.add(pair)

        clean_src.append(s)
        clean_tgt.append(t)
    print()
    print(f"Removed empty      : {removed_empty:,}")
    print(f"Removed too short  : {removed_short:,}")
    print(f"Removed duplicates : {removed_duplicates:,}")
    print(f"Final examples     : {len(clean_src):,}")

    return clean_src, clean_tgt

In [ ]:
# Apply cleaning

lang8_src, lang8_tgt = clean(lang8_src, lang8_tgt)
bea_src, bea_tgt = clean(bea_src, bea_tgt)
fce_src, fce_tgt = clean(fce_src, fce_tgt)


Removed empty      : 0
Removed too short  : 2,379
Removed duplicates : 84,735
Final examples     : 950,447

Removed empty      : 4
Removed too short  : 17
Removed duplicates : 809
Final examples     : 33,478

Removed empty      : 4
Removed too short  : 26
Removed duplicates : 2,622
Final examples     : 25,698


In [ ]:
# Combine datasets

train_src = lang8_src + bea_src + fce_src
train_tgt = lang8_tgt + bea_tgt + fce_tgt

In [ ]:
print(len(train_src))
print(len(train_tgt))

1009623
1009623


In [ ]:
# shuffle

import random

data = list(zip(train_src, train_tgt))
random.shuffle(data)

train_src, train_tgt = zip(*data)
train_src = list(train_src)
train_tgt = list(train_tgt)

In [ ]:
# number of examples that actually need correction
changed = 0
unchanged = 0

for s, t in zip(train_src, train_tgt):

    if s == t:
        unchanged += 1
    else:
        changed += 1

print(f"Changed   : {changed:,}")
print(f"Unchanged : {unchanged:,}")
print(f"Changed ratio: {changed / (changed + unchanged):.2%}")
print(f"Correction rate : {changed / len(train_src):.2%}")
print(f"Copy rate       : {unchanged / len(train_src):.2%}")

Changed   : 536,498
Unchanged : 473,125
Changed ratio: 53.14%
Correction rate : 53.14%
Copy rate       : 46.86%


In [ ]:
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer
)
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)


In [ ]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  20% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G  4.0K  5.7G   1% /dev/shm
/dev/root       2.0G  1.3G  696M  65% /usr/sbin/docker-init
/dev/sda1       114G   22G   93G  20% /kaggle/input
tmpfs           6.4G  3.9M  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive            15G   12G  3.5G  78% /content/drive


In [ ]:
!ls ~/.cache/huggingface/hub

models--t5-base


In [ ]:
# load T-5 base
!rm -rf ~/.cache/huggingface/hub/models--t5-base
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [ ]:
from datasets import Dataset
from backend.validation import validate
import random

TARGET_SIZE = 425_000

CHANGED_RATIO = 0.5314

TARGET_CHANGED = int(TARGET_SIZE * CHANGED_RATIO)
TARGET_UNCHANGED = TARGET_SIZE - TARGET_CHANGED

changed = []
unchanged = []

for s, t in zip(train_src, train_tgt):

    if s.strip() == t.strip():
        unchanged.append((s, t))
    else:
        changed.append((s, t))

print(f"Changed: {len(changed):,}")
print(f"Unchanged: {len(unchanged):,}")

random.seed(42)

changed = random.sample(changed, TARGET_CHANGED)
unchanged = random.sample(unchanged, TARGET_UNCHANGED)

samples = changed + unchanged
random.shuffle(samples)

train_src = [s for s, _ in samples]
train_tgt = [t for _, t in samples]

print(f"Final dataset: {len(train_src):,}")

dataset = Dataset.from_dict({
    "src": train_src,
    "tgt": train_tgt
})

validate(dataset)


In [ ]:
# Preprocess data

def preprocessing(example):
  inputs = ["grammar: " + item for item in example["src"]]

  model_inputs = tokenizer(
      inputs,
      max_length=128,
      padding=True,
      truncation=True,
  )

  labels = tokenizer(
    example["tgt"],
    max_length=128,
    padding=True,
    truncation=True
)

  label_ids = labels["input_ids"]

  label_ids = [
      [(token if token != tokenizer.pad_token_id else -100) for token in seq]
      for seq in label_ids
  ]

  model_inputs["labels"] = label_ids
  return model_inputs

In [ ]:
# Apply tokenization to model

tokenized_dataset = dataset.map(preprocessing, batched=True)

In [ ]:
split = tokenized_dataset.train_test_split(test_size=0.1)

train_dataset = split['train']
eval_dataset = split['test']

# train/validaition overlapping check
train_pairs = {
    (x["src"], x["tgt"])
    for x in train_dataset
}

eval_pairs = {
    (x["src"], x["tgt"])
    for x in eval_dataset
}

overlap = train_pairs & eval_pairs

print(f"Train examples      : {len(train_pairs):,}")
print(f"Validation examples : {len(eval_pairs):,}")
print(f"Overlapping pairs   : {len(overlap):,}")

In [ ]:
import evaluate

bleu = evaluate.load("bleu")

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    labels = np.where(
        labels == -100,
        tokenizer.pad_token_id,
        labels
    )

    predictions = np.where(
        predictions == -100,
        tokenizer.pad_token_id,
        predictions
    )

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    result = bleu.compute(
        predictions=decoded_preds,
        references=[[label] for label in decoded_labels]
    )

    return {
        "bleu": result["bleu"]
    }

In [ ]:
from datasets.utils import logging
import torch
eval_dataset = eval_dataset.shuffle(seed=42).select(range(5000))
!mkdir "/content/drive/MyDrive/GoogleColab/EnChatbot/backend/checkpoints"
OUTPUT_DIR = "/content/drive/MyDrive/GoogleColab/EnChatbot/backend/checkpoints"

# Debug args
# training_args = Seq2SeqTrainingArguments(
#     output_dir=OUTPUT_DIR,
#     eval_strategy="steps",
#     save_strategy="steps",
#     warmup_steps=500,
#     eval_steps=2000,
#     save_steps=2000,
#     save_total_limit=1,
#     learning_rate=3e-5,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=2,
#     per_device_eval_batch_size=4,
#     num_train_epochs=5,
#     weight_decay=0.01,
#     logging_steps=200,
#     fp16=torch.cuda.is_available(),
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     predict_with_generate=True,
#     greater_is_better=False
# )

#1M+ args
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="steps",
    save_strategy="steps",

    eval_steps=10000,
    save_steps=10000,

    save_total_limit=2,

    learning_rate=3e-5,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=2,

    eval_accumulation_steps=1,

    num_train_epochs=3,

    warmup_steps=2000,

    weight_decay=0.01,

    logging_steps=200,

    fp16=torch.cuda.is_available(),

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    predict_with_generate=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True
)

In [ ]:

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator

)

In [ ]:
# Train and save model
import os
from transformers.trainer_utils import get_last_checkpoint
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer # Import necessary classes

OUTPUT_DIR = "/content/drive/MyDrive/GoogleColab/EnChatbot/backend/checkpoints"

last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is None:
  trainer.train()

else:
  trainer.train(
      resume_from_checkpoint=last_checkpoint)

model.save_pretrained('/content/drive/MyDrive/GoogleColab/EnChatbot/backend/models')
tokenizer.save_pretrained('/content/drive/MyDrive/GoogleColab/EnChatbot/backend/models')

# deleting files in OUT_DIR to free up drive space
import shutil
shutil.rmtree(OUTPUT_DIR)

In [ ]:
# from transformers import T5ForConditionalGeneration, T5Tokenizer
# import torch

# MODEL_PATH = "/content/drive/MyDrive/GoogleColab/EnChatbot/backend/models"

# tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH)

# model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)

# device = "cuda" if torch.cuda.is_available() else "cpu"

# model.to(device)
# model.eval()

In [17]:
%cd /content/drive/MyDrive/GoogleColab/EnChatbot

/content/drive/MyDrive/GoogleColab/EnChatbot


In [18]:
!dir

backend  datasets  debug(1).ipynb  final\ project.ipynb  frontend  requirement


In [19]:
!git init
!git branch -M main

Reinitialized existing Git repository in /content/drive/MyDrive/GoogleColab/EnChatbot/.git/


In [20]:
!git config --global user.name "lurfe"
!git config --global user.email "khuongle2k21@gmail.com"

In [21]:
!git remote add origin https://github.com/lurfe/EnChatBot.git

error: remote origin already exists.


In [22]:
!git remote -v

origin	https://github.com/lurfe/EnChatBot.git (fetch)
origin	https://github.com/lurfe/EnChatBot.git (push)


In [32]:
%%writefile .gitignore
__pycache__/
*.pyc

.ipynb_checkpoints/

logs/
checkpoints/
models/
datasets/

*.pt
*.pth
*.bin
*.safetensors

.env

Overwriting .gitignore


In [33]:
!cat .gitignore

__pycache__/
*.pyc

.ipynb_checkpoints/

logs/
checkpoints/
models/
datasets/

*.pt
*.pth
*.bin
*.safetensors

.env


In [28]:
!pwd
!dir -a

/content/drive/MyDrive/GoogleColab/EnChatbot
backend   debug(1).ipynb	frontend  .gitignore	      requirement
datasets  final\ project.ipynb	.git	  .ipynb_checkpoints


In [29]:
!git add .

In [30]:
!du -sh datasets

33M	datasets


In [34]:
!git status

Refresh index: 100% (29/29), done.
On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitignore
	new file:   backend/__init__.py
	new file:   backend/app.py
	new file:   backend/config.py
	new file:   backend/debug.py
	new file:   backend/logger.py
	new file:   backend/services/__init__.py
	new file:   backend/services/conversation.py
	new file:   backend/services/edits.py
	new file:   backend/services/enums.py
	new file:   backend/services/explanation.py
	new file:   backend/services/grammar.py
	new file:   backend/services/grammar_rules.py
	new file:   backend/services/llama.py
	new file:   backend/services/memory.py
	new file:   backend/services/pipeline.py
	new file:   backend/services/prompt.py
	new file:   backend/services/router.py
	new file:   backend/services/schemas.py
	new file:   backend/services/splitter.py
	new file:   backend/splitter.py
	new file:   backend/utils/helpers.py
	new file:   backend/utils/tim

In [38]:
!git rm --cached -r -f datasets

rm 'datasets/lang-8/lang8.bea19.tar.gz'


In [40]:
!git add .gitignore
!git add "final project.ipynb"
!git add

In [41]:
!git status

On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitignore
	new file:   backend/__init__.py
	new file:   backend/app.py
	new file:   backend/config.py
	new file:   backend/debug.py
	new file:   backend/logger.py
	new file:   backend/services/__init__.py
	new file:   backend/services/conversation.py
	new file:   backend/services/edits.py
	new file:   backend/services/enums.py
	new file:   backend/services/explanation.py
	new file:   backend/services/grammar.py
	new file:   backend/services/grammar_rules.py
	new file:   backend/services/llama.py
	new file:   backend/services/memory.py
	new file:   backend/services/pipeline.py
	new file:   backend/services/prompt.py
	new file:   backend/services/router.py
	new file:   backend/services/schemas.py
	new file:   backend/services/splitter.py
	new file:   backend/splitter.py
	new file:   backend/utils/helpers.py
	new file:   backend/utils/timer.py
	new file:   backend/utils/to